# Multi-turn conversations with tools

One question can need **several tool calls in a row**. "What day is 103 days from today?":

| # | role | content |
|---|---|---|
| 1 | user | What day is 103 days from today? |
| 2 | assistant | `tool_use`: `get_current_datetime` |
| 3 | user | `tool_result`: "2026-10-29" |
| 4 | assistant | `tool_use`: `add_duration_to_datetime("2026-10-29", 103, "days")` |
| 5 | user | `tool_result`: "Thursday, February 09, 2027 ..." |
| 6 | assistant | text: the answer |

Claude cannot ask for the second tool before it has the first result: it needs today's date as input.
Your code does not know in advance how many rounds there will be, so it has to repeat until
Claude stops asking. The lesson gives the loop as pseudocode:

```python
def run_conversation(messages):
    while True:
        response = chat(messages)
        add_assistant_message(messages, response)
        if response isn't asking for a tool:  # pseudocode
            break
        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)
    return messages
```

The helpers it relies on (`chat` returns the whole `Message`, `add_*_message` take blocks)
were added in `03_handling_message_blocks`. The real loop is the next lesson; here the rounds are
run one by one to see what happens in each.

## Setup

Tool functions and schemas live in [`tools.py`](./tools.py): `get_current_datetime` from the first
lessons, `add_duration_to_datetime` and `set_reminder` as the course provides them.

In [1]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
from tools import (
    add_duration_to_datetime,
    add_duration_to_datetime_schema,
    get_current_datetime,
    get_current_datetime_schema,
)

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

## The new tool

Date arithmetic is exactly what language models get wrong, so it is done in code.

In [2]:
print(add_duration_to_datetime("2026-10-29", 103))
print(add_duration_to_datetime("2026-01-31", 1, unit="months"))

Tuesday, February 09, 2027 12:00:00 AM
Saturday, February 28, 2026 12:00:00 AM


## Run the rounds one by one

`next_round` does one pass of the pseudocode loop: ask Claude, and if it requests a tool,
run it and add the result to the history. `tool_functions` maps a tool name to the Python
function to call.

In [3]:
tools = [get_current_datetime_schema, add_duration_to_datetime_schema]
tool_functions = {
    "get_current_datetime": get_current_datetime,
    "add_duration_to_datetime": add_duration_to_datetime,
}


def next_round(messages):
    response = chat(messages, tools=tools)
    add_assistant_message(messages, response)

    text = text_from_message(response)
    if text:
        print("Claude says:", text)

    if response.stop_reason != "tool_use":
        print("Done, stop_reason:", response.stop_reason)
        return

    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        result = tool_functions[block.name](**block.input)
        print(f"Claude asks: {block.name}({block.input})")
        print(f"We answer:   {result}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": result,
            "is_error": False,
        })
    add_user_message(messages, tool_results)


messages = []
add_user_message(messages, "What day is 103 days from today?")

In [4]:
next_round(messages)  # round 1

Claude says: I'll help you find out what day it will be 103 days from today.
Claude asks: get_current_datetime({'date_format': '%Y-%m-%d'})
We answer:   2026-10-29


In [5]:
next_round(messages)  # round 2

Claude says: Now let me add 103 days to today's date:
Claude asks: add_duration_to_datetime({'datetime_str': '2026-10-29', 'duration': 103, 'unit': 'days'})
We answer:   Tuesday, February 09, 2027 12:00:00 AM


In [6]:
next_round(messages)  # round 3

Claude says: 103 days from today (October 29, 2026) will be **Tuesday, February 9, 2027**.
Done, stop_reason: end_turn


## The whole history

In [7]:
for number, message in enumerate(messages, start=1):
    content = message["content"]
    if isinstance(content, str):
        summary = repr(content)
    else:
        summary = ", ".join(
            block["type"] if isinstance(block, dict) else block.type for block in content
        )
    print(f"{number}. {message['role']:9} {summary}")

1. user      'What day is 103 days from today?'
2. assistant text, tool_use
3. user      tool_result
4. assistant text, tool_use
5. user      tool_result
6. assistant text


Three identical cells, and the number of rounds was only known after running them.
Next lesson: a `while` loop instead, plus `run_tools` with error handling (`is_error`).